## **PART 5. DATA MODELLING**

In [1]:
#import library
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv("./data_processed.csv")

In [3]:
data.head()

,Product Name,Category,Dosage Form,Price,Trademark,Brand Origin,Country,Rating,Continent,General_function
0,"Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...",Chăm sóc cơ thể,Gel,105000.0,DECUMAR,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể
1,Dung dịch vệ sinh vùng kín Bimunica 250ml dành...,Chăm sóc cơ thể,Viên nén,230000.0,Kingphar,Hoa Kỳ,Liên Bang Nga,5.0,Europe,Chăm sóc cơ thể
2,"Kem giảm thâm vùng nách, mông, bikini Neothera...",Chăm sóc cơ thể,Viên nén,139000.0,La Beauty,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể
3,Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...,Chăm sóc cơ thể,Viên nén,390000.0,SVR,Pháp,Pháp,unknown,Europe,Chăm sóc cơ thể
4,Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...,"Lăn khử mùi, xịt khử mùi",Dạng bọt,96000.0,Kingphar,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể


In [4]:
data.shape

(1999, 10)

In [5]:
# Create a label encoder object
label_encoder = preprocessing.LabelEncoder()

# Encode labels in the 'Country' column
data['Country'] = label_encoder.fit_transform(data['Country'])
data['Trademark'] = label_encoder.fit_transform(data['Trademark'])
data['General_function'] = label_encoder.fit_transform(data['General_function'])

print(data.head())

                                        Product Name  \
0  Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...   
1  Dung dịch vệ sinh vùng kín Bimunica 250ml dành...   
2  Kem giảm thâm vùng nách, mông, bikini Neothera...   
3  Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...   
4  Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...   

                   Category Dosage Form     Price  Trademark Brand Origin  \
0           Chăm sóc cơ thể         Gel  105000.0         75     Việt Nam   
1           Chăm sóc cơ thể    Viên nén  230000.0        208       Hoa Kỳ   
2           Chăm sóc cơ thể    Viên nén  139000.0        217     Việt Nam   
3           Chăm sóc cơ thể    Viên nén  390000.0        375         Pháp   
4  Lăn khử mùi, xịt khử mùi    Dạng bọt   96000.0        208     Việt Nam   

   Country   Rating Continent  General_function  
0       38      5.0      Asia                 0  
1       16      5.0    Europe                 0  
2       38      5.0      Asia                 0  


In [6]:
train_data = data[data['Rating'] != 'unknown']  # Product with rating
prediction_data = data[data['Rating'] == 'unknown']  # Product with rating = "unknown"

In [7]:
#Check shape of data
train_data.shape, prediction_data.shape

((1220, 10), (779, 10))

### 2. Feature Selection

In [8]:
#These will be the independent variables
features = ['Price', 'Trademark', 'Country', 'General_function']

### 3. Splitting dataset into X and y

In [9]:
# Data for training and test
X = train_data[features]
y = train_data["Rating"]
# Data for prediction
X_test = prediction_data[features]
y_test = prediction_data["Rating"]

In [10]:
#Check
X.head()

,Price,Trademark,Country,General_function
0,105000.0,75,38,0
1,230000.0,208,16,0
2,139000.0,217,38,0
4,96000.0,208,38,0
5,132000.0,208,11,1


In [11]:
#Check
y.head()

0    5.0
1    5.0
2    5.0
4    5.0
5    5.0
Name: Rating, dtype: object

##### X,y -> X_train, y_train, X_valid, y_valid

In [12]:
from sklearn.model_selection import train_test_split
# Split train and test model in 80/20
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state=0)

In [13]:
X.shape, X_train.shape, X_valid.shape

((1220, 4), (976, 4), (244, 4))

### **Model training**

To choose an algorithm suitable for the data, we refer to instructions from the official Scikit-learn documentation, including the following criteria:
1. Type of problem:
    * This is a problem of predicting continuous value (the amount of customer reviews), so it belongs to the group of regression problems.
2. Data scale:
    * With the current data size less than 100K samples, we choose algorithms suitable for small or medium data.
    
-> Considered regression algorithm:

- Random Forest Regression: Powerful in handling complex data and does not require data normalization.
- Gradient Boosting (XGBoost): Effective for problems that require accurate prediction, capable of handling outliers well.
- Ridge Regression: Suitable for small data, add regularization to avoid overfitting.
- SVR(kernel=rbf): Good for small problems that require accurate predictions, but can be slow when the data is large.
- ElasticNet Regression: Combines L1 and L2 regularization, handles well with data with many unrelated features.

In [14]:
#import library
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,  make_scorer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

### **Random Forest algorithm**

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd

parameters = {
    'n_estimators' : [100, 200, 300, 400],
    'max_depth': [1, 2, 3, 4]
}

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Train the model
rf_regressor = RandomForestRegressor(random_state=0)
clf = GridSearchCV(rf_regressor, parameters)
clf.fit(X_train_scaled, y_train)

# Get the best hyperparameters
best_params = clf.best_params_
print("Best Parameters:", best_params)

# Initialize the model with the best hyperparameters
rf_best = RandomForestRegressor(n_estimators=best_params['n_estimators'],
                                max_depth=best_params['max_depth'],
                                random_state=0)

# Retrain the model with the best hyperparameters
rf_best.fit(X_train_scaled, y_train)

# Predict with the optimized model
y_pred_best = rf_best.predict(X_valid_scaled)

y_valid = pd.to_numeric(y_valid, errors='coerce')

# Evaluate the model
mae_best = mean_absolute_error(y_valid, y_pred_best)
mse_best = mean_squared_error(y_valid, y_pred_best)

mape = np.mean(np.abs((y_valid - y_pred_best) / y_valid)) * 100
accuracy_best = 100 - mape

print(f"MAE (Best Model): {mae_best}")
print(f"MSE (Best Model): {mse_best}")
print(f"Score (Best Model): {round(accuracy_best, 2)}%")

# Display results
results_best = pd.DataFrame(zip(y_valid, y_pred_best, y_valid - y_pred_best), 
                            columns=['y_valid', 'y_pred', 'error'])

print(results_best.head(10))


Best Parameters: {'max_depth': 1, 'n_estimators': 100}
MAE (Best Model): 0.1426799939014015
MSE (Best Model): 0.06863167918397933
Score (Best Model): 96.76%
   y_valid    y_pred     error
0      5.0  4.893403  0.106597
1      5.0  4.918376  0.081624
2      5.0  4.897346  0.102654
3      5.0  4.898596  0.101404
4      5.0  4.918290  0.081710
5      5.0  4.917630  0.082370
6      5.0  4.915519  0.084481
7      5.0  4.918292  0.081708
8      5.0  4.892110  0.107890
9      5.0  4.893405  0.106595


### **Gradient Boosting algorithm**

In [16]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Hyperparameter grid for Gradient Boosting
gb_parameters = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [1, 2, 3, 4]
}

gb_regressor = GradientBoostingRegressor(random_state=0)
gb_clf = GridSearchCV(gb_regressor, gb_parameters, scoring='neg_mean_absolute_error', cv=3)
gb_clf.fit(X_train_scaled, y_train)

# Get best parameters and refit Gradient Boosting
best_gb_params = gb_clf.best_params_
print("Best Parameters (Gradient Boosting):", best_gb_params)
gb_best = GradientBoostingRegressor(n_estimators=best_gb_params['n_estimators'],
                                    learning_rate=best_gb_params['learning_rate'],
                                    max_depth=best_gb_params['max_depth'],
                                    random_state=0)
gb_best.fit(X_train_scaled, y_train)
gb_y_pred = gb_best.predict(X_valid_scaled)

# Evaluate Gradient Boosting model
gb_mae = mean_absolute_error(y_valid, gb_y_pred)
gb_mse = mean_squared_error(y_valid, gb_y_pred)
gb_mape = np.mean(np.abs((y_valid - gb_y_pred) / y_valid)) * 100
gb_accuracy = 100 - gb_mape

print(f"Gradient Boosting - MAE: {gb_mae}")
print(f"Gradient Boosting - MSE: {gb_mse}")
print(f"Gradient Boosting - Accuracy: {gb_accuracy:.2f}%")

# Display the results for Gradient Boosting
gb_results = pd.DataFrame(
    zip(y_valid, gb_y_pred, y_valid - gb_y_pred),
    columns=['y_valid', 'y_pred', 'error']
)

# Display the first 10 rows of results
gb_results.head(10)

Best Parameters (Gradient Boosting): {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 100}
Gradient Boosting - MAE: 0.15135671934721198
Gradient Boosting - MSE: 0.08634435943253711
Gradient Boosting - Accuracy: 96.59%


,y_valid,y_pred,error
0,5.0,4.885467,0.114533
1,5.0,4.930805,0.069195
2,5.0,4.885828,0.114172
3,5.0,4.888064,0.111936
4,5.0,4.915754,0.084246
5,5.0,4.930805,0.069195
6,5.0,4.915579,0.084421
7,5.0,4.916096,0.083904
8,5.0,4.883748,0.116252
9,5.0,4.885252,0.114748


### **Ridge Regression algorithm**

In [17]:
from sklearn.preprocessing import PolynomialFeatures

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Find the optimal alpha using GridSearchCV
params = {'alpha': [0.01, 0.1, 1, 10, 100]}
ridge_cv = GridSearchCV(Ridge(), params, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)

# Get the optimal alpha
best_alpha = ridge_cv.best_params_['alpha']
print("Best alpha:", best_alpha)

# Enhance the model's ability to learn non-linear relationships
poly = PolynomialFeatures(degree=10)
X_train_poly = poly.fit_transform(X_train_scaled)
X_valid_poly = poly.transform(X_valid_scaled)

# Retrain the model with the optimal alpha
rid_reg_poly = Ridge(alpha=best_alpha)
rid_reg_poly.fit(X_train_poly, y_train)
y_pred_poly = rid_reg_poly.predict(X_valid_poly)

# Evaluate the model
r2 = r2_score(y_valid, y_pred_poly)
mae = mean_absolute_error(y_valid, y_pred_poly)
mse = mean_squared_error(y_valid, y_pred_poly)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_valid - y_pred_poly) / y_valid)) * 100
accuracy_best = 100 - mape

print("R^2 Score:", r2)
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print(f"Score (Best Model): {round(accuracy_best, 2)}%")
#pd.DataFrame({'y' : y_valid.head(), 'y_preds': y_pred_poly})


Best alpha: 100


c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(


R^2 Score: -1.2126029174029687
Mean Absolute Error (MAE): 0.16258196836810943
Mean Squared Error (MSE): 0.15065850223011915
Root Mean Squared Error (RMSE): 0.38814752637382494
Score (Best Model): 96.37%


### **SVR(kernel=rbf)**

In [18]:
#YOUR CODE HERE
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Function to standardize the data
def preprocess_data(X_train, X_valid):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)
    return X_train_scaled, X_valid_scaled

# Function to evaluate the model
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"R² Score: {r2:.4f}")
    return mae, mse, r2

# 1. Standardize the data
X_train_scaled, X_valid_scaled = preprocess_data(X_train, X_valid)

# 2. Initialize and train the model
kernel = 'rbf'  # Kernel for SVR (can try 'linear' or 'poly')
C = 1.0         # Regularization parameter
epsilon = 0.1   # Epsilon in the SVR model

svr_model = SVR(kernel=kernel, C=C, epsilon=epsilon)
svr_model.fit(X_train_scaled, y_train)

# 3. Predict and evaluate the model
y_pred = svr_model.predict(X_valid_scaled)
evaluate_model(y_valid, y_pred)

mape = np.mean(np.abs((y_valid - y_pred) / y_valid)) * 100
accuracy_best = 100 - mape
print(f"Score (Best Model): {round(accuracy_best, 2)}%")


MAE: 0.1444
MSE: 0.0683
R² Score: -0.0035
Score (Best Model): 96.73%


### **ElasticNet**

In [19]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Search for the optimal alpha and l1_ratio parameters using GridSearchCV
param_grid = {
    'alpha': np.logspace(-4, 1, 10),  # Search alpha from 0.0001 to 10
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]  # Search l1_ratio from 0 to 1
}
elastic_net = ElasticNet(random_state=0)

# GridSearchCV to find the optimal parameters
grid_search = GridSearchCV(estimator=elastic_net, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=1)
grid_search.fit(X_train_scaled, y_train)

# Get the optimal parameters
best_alpha = grid_search.best_params_['alpha']
best_l1_ratio = grid_search.best_params_['l1_ratio']
print(f"Best alpha: {best_alpha}")
print(f"Best l1_ratio: {best_l1_ratio}")

# Retrain the model with the optimal alpha and l1_ratio
elastic_net_best = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio, random_state=0)
elastic_net_best.fit(X_train_scaled, y_train)
y_pred = elastic_net_best.predict(X_valid_scaled)

# Evaluate the model
r2 = r2_score(y_valid, y_pred)
mae = mean_absolute_error(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_valid - y_pred) / y_valid)) * 100
accuracy_best = 100 - mape

print("R^2 Score:", r2)
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print(f"Score (Best Model): {round(accuracy_best, 2)}%")


Best alpha: 0.05994842503189409
Best l1_ratio: 0.5
R^2 Score: -0.003609051350930814
Mean Absolute Error (MAE): 0.14345941951088378
Mean Squared Error (MSE): 0.06833681512026334
Root Mean Squared Error (RMSE): 0.261413111989937
Score (Best Model): 96.75%


### **Cross-validation** 

Khi so sánh các thuật toán mô hình hóa, việc tránh bias giữa các thuật toán là rất quan trọng. Để đánh giá các mô hình một cách khách quan và chính xác nhất, chúng ta sử dụng phương pháp Cross-Validation (K-fold), được cung cấp bởi Scikit-learn.

Phương pháp này hoạt động như sau:

- Chia tập dữ liệu huấn luyện thành K phần (folds) bằng nhau.
- Với mỗi lần lặp, giữ lại 1 fold để làm tập kiểm tra (validation set), và sử dụng K−1 folds còn lại để huấn luyện mô hình.
- Sau mỗi lần lặp, tính các metrics như RMSE (Root Mean Squared Error), MSE (Mean Squared Error).
- Sau khi hoàn thành K lần lặp, tổng hợp kết quả của từng fold và tính trung bình cộng (mean) của các metrics.
- Giá trị trung bình này được sử dụng để đánh giá hiệu suất mô hình một cách toàn diện và so sánh với các mô hình khác.

In [20]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
import time
import tracemalloc

# Fix the random seed to ensure reproducible results
seed = 2024

# Create Random Forest Regressor model
rf_model = RandomForestRegressor(
    n_estimators=best_params['n_estimators'],  # Number of trees in the forest
    max_depth=best_params['max_depth'],       # Maximum depth of each tree
    random_state=seed                         # Ensure reproducible results
)
rf_model.__class__.__name__ = "RandomForestRegression"  # Rename for easier display

# Create pipeline for Ridge Regression with polynomial features
ridge_poly_pipeline = Pipeline([
    ('scaler', StandardScaler()),              # Standardize the data
    ('poly', PolynomialFeatures(degree=2)),    # Add polynomial features (degree 2)
    ('ridge', Ridge(alpha=best_alpha))         # Ridge Regression with optimal alpha
])  
ridge_poly_pipeline.__class__.__name__ = "RidgeRegression"  # Rename for easier display

# Create Gradient Boosting Regressor model
GradientBoosting_model = GradientBoostingRegressor(
    n_estimators=best_gb_params['n_estimators'],  # Number of trees
    learning_rate=best_gb_params['learning_rate'], # Learning rate
    max_depth=best_gb_params['max_depth'],         # Maximum depth
    random_state=seed                              # Ensure reproducible results
)
GradientBoosting_model.__class__.__name__ = "GradientBoostingRegression"

# Create ElasticNet Regressor model
Elastic_model = ElasticNet(
    alpha=best_alpha,          # L2 regularization parameter
    l1_ratio=best_l1_ratio,    # Ratio between L1 and L2
    random_state=seed          # Ensure reproducible results
)
Elastic_model.__class__.__name__ = "ElasticNetRegression"

# Create Support Vector Regressor (SVR) model
svr_model = SVR(
    kernel=kernel,   # Kernel to use (e.g., 'rbf')
    C=C,             # Regularization parameter
    epsilon=epsilon  # Epsilon in the SVR model
)
svr_model.__class__.__name__ = "SVR(kernel=rbf)"

# List of models to compare
models = [
    rf_model,
    ridge_poly_pipeline,
    Elastic_model,
    GradientBoosting_model,
    svr_model
]

# RMSE (Root Mean Square Error) scorer
rmse_scorer = make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=False)

# MSE (Mean Squared Error) scorer
mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

# Function to evaluate models
def generate_baseline_results(models, X, y, cv=5):
    kfold = KFold(n_splits=cv, shuffle=True, random_state=seed)  # K-Fold Cross Validation
    entries = []  # Store results of each model

    for model in models:
        model_name = model.__class__.__name__  # Model name
        rmse_scores = []  # Store RMSE values for each fold
        mse_scores = []   # Store MSE values for each fold
        times = []        # Store training times
        memory_usages = []  # Store memory usage

        for train_idx, valid_idx in kfold.split(X, y):
            # Split data into train and validation sets
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            # Start tracking time and memory
            start_time = time.time()  # Record start time
            tracemalloc.start()      # Start memory tracking

            # Train the model
            model.fit(X_train, y_train)

            # Predict on validation set
            y_pred = model.predict(X_valid)

            # Calculate evaluation metrics
            rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
            mse = mean_squared_error(y_valid, y_pred)

            # Get memory usage information
            current, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            # Calculate training time
            elapsed_time = time.time() - start_time

            # Store results
            rmse_scores.append(rmse)
            mse_scores.append(mse)
            times.append(elapsed_time)
            memory_usages.append(peak / 1024 / 1024)  # Convert to MB

        # Calculate average results
        entries.append({
            'model_name': model_name,
            'mean_rmse': np.mean(rmse_scores),          # Average RMSE
            'mean_mse': np.mean(mse_scores),            # Average MSE
            'mean_time': np.mean(times),                # Average training time
            'mean_memory_usage': np.mean(memory_usages) # Average memory usage (MB)
        })

    # Convert results to DataFrame
    baseline_results = pd.DataFrame(entries)
    baseline_results.sort_values(by=['mean_rmse'], ascending=True, inplace=True)  # Sort by RMSE

    return baseline_results

# Call the function to evaluate models
generate_baseline_results(models, X, y, cv=5)


,model_name,mean_rmse,mean_mse,mean_time,mean_memory_usage
4,SVR(kernel=rbf),0.340779,0.130761,0.016284,0.101542
2,ElasticNetRegression,0.341843,0.131333,0.008504,0.107597
1,RidgeRegression,0.342702,0.131952,0.012901,0.312237
0,RandomForestRegression,0.342813,0.132203,0.588857,0.177014
3,GradientBoostingRegression,0.346877,0.133857,0.162812,0.176443


### **Choose the best model and predict**

- The best model is selected based on the lowest Mean RMSE and MSE values ​​from the Cross-Validation results.
- After choosing the best model, retrain the model on the entire data set to increase accuracy, then predict **Rating** for products without information and save the results Go to the new column **Predicted Rating**
    

In [21]:
baseline_results = generate_baseline_results(models, X, y, cv=5)

# Select the best model
best_model_name = baseline_results.iloc[0]['model_name']  # Get the name of the model with the smallest Mean RMSE
print(f"\nBest Model Selected: {best_model_name}")

if best_model_name == "Ridge":
    # If the best model is Ridge 
    best_model = ridge_poly_pipeline
else:
    # If the best model is RandomForestRegressor
    best_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=seed)
    
# Retrain the best model on the entire training set
best_model.fit(X, y)

# Predict ratings for the prediction_data set
predictions = best_model.predict(X_test)

# Output the prediction results
prediction_data['Predicted_Rating'] = predictions
print("\nPredicted Ratings for 'unknown':")
print(prediction_data[['Price', 'Trademark', 'Country', 'General_function', 'Predicted_Rating']])

# Save the results to a file 
prediction_data.to_csv("predicted_ratings.csv", index=False)



Best Model Selected: SVR(kernel=rbf)

Predicted Ratings for 'unknown':
         Price  Trademark  Country  General_function  Predicted_Rating
3     390000.0        375       23                 0          4.944464
10     22400.0        214       38                 3          4.919375
14    151200.0        290       38                 0          4.968480
30     39000.0         14       38                 0          4.975479
31     39000.0         14       38                 0          4.975479
...        ...        ...      ...               ...               ...
1994  230000.0        208       38                14          4.974350
1995  230000.0        342       38                 4          4.790315
1996  230000.0        342       38                15          4.554150
1997  230000.0        234        7                 5          4.940933
1998  230000.0        157        8                15          4.800758

[779 rows x 5 columns]


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30160\2677336469.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prediction_data['Predicted_Rating'] = predictions
